In [1]:
import time
import torch
from torch.distributions import Categorical, kl
from d2l.torch import Animator

from net import Net
from aco import ACO
from utils import gen_pyg_data, load_test_dataset

torch.manual_seed(12345)

EPS = 1e-10
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [2]:
import torch

# 打印当前分配的设备
print("当前代码正在使用的设备是:", device)

# 进一步检查显卡状态
if torch.cuda.is_available():
    print("✅ 成功检测到独立显卡！")
    print("显卡型号:", torch.cuda.get_device_name(0))
else:
    print("❌ 警告：未检测到可用的独立显卡，系统正在使用 CPU 进行计算。")

当前代码正在使用的设备是: cuda:0
✅ 成功检测到独立显卡！
显卡型号: NVIDIA GeForce RTX 3060 Laptop GPU


In [6]:
@torch.no_grad()
def infer_instance(model, pyg_data, distances, n_ants, t_aco_diff, k_sparse=None):
    if model:
        model.eval()
        heu_vec = model(pyg_data)
        heu_mat = model.reshape(pyg_data, heu_vec) + EPS
        aco = ACO(n_ants=n_ants, heuristic=heu_mat, distances=distances, device=device)
    else:
        aco = ACO(n_ants=n_ants, distances=distances, device=device)
        if k_sparse:
            aco.sparsify(k_sparse)
        
    results = torch.zeros(size=(len(t_aco_diff),), device=device)
    for i, t in enumerate(t_aco_diff):
        best_cost = aco.run(t)
        results[i] = best_cost
    return results
        
@torch.no_grad()
def test(dataset, model, n_ants, t_aco, k_sparse=None):
    _t_aco = [0] + t_aco
    t_aco_diff = [_t_aco[i+1]-_t_aco[i] for i in range(len(_t_aco)-1)]
    sum_results = torch.zeros(size=(len(t_aco_diff),), device=device)
    
    total = len(dataset)
    print(f"🚀 测试正式开始！总共有 {total} 个测试实例，请稍候...", flush=True)
    
    start = time.time()
    for i, (pyg_data, distances) in enumerate(dataset):
        results = infer_instance(model, pyg_data, distances, n_ants, t_aco_diff, k_sparse)
        sum_results += results
        
        # 改成每 10 个打印一次，并且强制立即输出在屏幕上
        if (i + 1) % 50 == 0:
            print(f"-> 正在拼命计算... 已完成: {i + 1} / {total}", flush=True)
            
    end = time.time()
    print(f"✅ 测试全部完成！总耗时: {end-start:.2f} 秒", flush=True)
    return sum_results / total, end-start

### Test on TSP20

DeepACO

In [7]:
n_ants = 20
n_node = 20
k_sparse = 10
t_aco = [1, 10, 20, 30, 40, 50, 100]
test_list = load_test_dataset(n_node, k_sparse, device)

# 🌟 关键修改：我们只取前 100 个实例进行测试，节约 90% 的时间！
test_list = test_list[:100] 

net_tsp = Net().to(device)
net_tsp.load_state_dict(torch.load(f'../pretrained/tsp/tsp{n_node}.pt', map_location=device))
avg_aco_best, duration = test(test_list, net_tsp, n_ants, t_aco, k_sparse)

print('total duration: ', duration)
for i, t in enumerate(t_aco):
    print("T={}, average cost is {}.".format(t, avg_aco_best[i]))

🚀 测试正式开始！总共有 500 个测试实例，请稍候...
-> 正在拼命计算... 已完成: 50 / 500
-> 正在拼命计算... 已完成: 100 / 500
-> 正在拼命计算... 已完成: 150 / 500
-> 正在拼命计算... 已完成: 200 / 500
-> 正在拼命计算... 已完成: 250 / 500
-> 正在拼命计算... 已完成: 300 / 500
-> 正在拼命计算... 已完成: 350 / 500
-> 正在拼命计算... 已完成: 400 / 500
-> 正在拼命计算... 已完成: 450 / 500
-> 正在拼命计算... 已完成: 500 / 500
✅ 测试全部完成！总耗时: 1459.01 秒
total duration:  1459.00586104393
T=1, average cost is 3.9779834747314453.
T=10, average cost is 3.858649969100952.
T=20, average cost is 3.848206043243408.
T=30, average cost is 3.844769239425659.
T=40, average cost is 3.842304229736328.
T=50, average cost is 3.8409512042999268.
T=100, average cost is 3.838022232055664.


ACO

In [8]:
n_ants = 20
n_node = 20
k_sparse = 10
t_aco = [1, 10, 20, 30, 40, 50, 100]
test_list = load_test_dataset(n_node, k_sparse, device)

# 🌟 关键修改：同样只取前 100 个实例保证对比公平
test_list = test_list[:100]

avg_aco_best, duration = test(test_list, None, n_ants, t_aco, k_sparse)

print('total duration: ', duration)
for i, t in enumerate(t_aco):
    print("T={}, average cost is {}.".format(t, avg_aco_best[i]))

🚀 测试正式开始！总共有 500 个测试实例，请稍候...
-> 正在拼命计算... 已完成: 50 / 500
-> 正在拼命计算... 已完成: 100 / 500
-> 正在拼命计算... 已完成: 150 / 500
-> 正在拼命计算... 已完成: 200 / 500
-> 正在拼命计算... 已完成: 250 / 500
-> 正在拼命计算... 已完成: 300 / 500
-> 正在拼命计算... 已完成: 350 / 500
-> 正在拼命计算... 已完成: 400 / 500
-> 正在拼命计算... 已完成: 450 / 500
-> 正在拼命计算... 已完成: 500 / 500
✅ 测试全部完成！总耗时: 1405.59 秒
total duration:  1405.5862710475922
T=1, average cost is 5.2567830085754395.
T=10, average cost is 4.245449542999268.
T=20, average cost is 4.060759544372559.
T=30, average cost is 3.9822607040405273.
T=40, average cost is 3.9355616569519043.
T=50, average cost is 3.905104875564575.
T=100, average cost is 3.8548083305358887.


In [5]:
import torch
import time
from aco import ACO
from utils import gen_pyg_data, load_val_dataset

# 1. 导入两个不同版本的网络架构
from net_orig import Net as NetOrig
from net_gat import Net as NetGAT

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

# ==========================================
# 准备相同的测试卷 (保证绝对公平)
# ==========================================
n_node = 20
k_sparse = 10
n_ants = 20
n_tests = 100 # 测试 100 个实例
print(f"正在加载 {n_tests} 个 TSP{n_node} 测试实例...")
dataset = load_val_dataset(n_node, k_sparse, device)

# ==========================================
# 加载两位选手的权重
# ==========================================
print("-> 正在加载【原始模型 (12层GCN)】...")
net_orig = NetOrig().to(device)
net_orig.load_state_dict(torch.load('./tsp20_orig.pt', map_location=device))
net_orig.eval() # 锁定权重和 BatchNorm

print("-> 正在加载【改进模型 (4层GATv2)】...")
net_gat = NetGAT().to(device)
net_gat.load_state_dict(torch.load('./tsp20_gatv2.pt', map_location=device))
net_gat.eval() # 锁定权重和 BatchNorm

# ==========================================
# 定义统一的打分裁判函数
# ==========================================
def evaluate_model(model, model_name):
    print(f"\n=========================================")
    print(f"🚀 考试开始: {model_name}")
    print(f"=========================================")
    start_time = time.time()
    
    cost_T1, cost_T10, cost_T100 = 0.0, 0.0, 0.0
    
    with torch.no_grad(): # 测试阶段关闭梯度计算
        for i, (pyg_data, distances) in enumerate(dataset[:n_tests]):
            # 神经网络推断启发式矩阵
            heu_vec = model(pyg_data)
            heu_mat = model.reshape(pyg_data, heu_vec) + 1e-10
            
            # 蚁群搜索
            aco = ACO(distances, n_ants, heuristic=heu_mat, device=device)
            
            # --- 修复核心：安全接收返回值 ---
            # T=1
            out = aco.run(1)
            costs = out[0] if isinstance(out, tuple) else out
            cost_T1 += costs.min().item() if costs.dim() > 0 else costs.item()
            
            # 累计到 T=10 (再跑9轮)
            out = aco.run(9)
            costs = out[0] if isinstance(out, tuple) else out
            cost_T10 += costs.min().item() if costs.dim() > 0 else costs.item()
            
            # 累计到 T=100 (再跑90轮)
            out = aco.run(90)
            costs = out[0] if isinstance(out, tuple) else out
            cost_T100 += costs.min().item() if costs.dim() > 0 else costs.item()
            # --------------------------------
            
            if (i+1) % 20 == 0:
                print(f"   进度: {i+1} / {n_tests}")

    total_time = time.time() - start_time
    print(f"\n✅ 【{model_name}】测试报告单:")
    print(f"⏱️ 总耗时: {total_time:.2f} 秒")
    print(f"🎯 T=1   平均路径: {cost_T1 / n_tests:.4f} (考量先验知识质量)")
    print(f"🎯 T=10  平均路径: {cost_T10 / n_tests:.4f} (考量早期收敛速度)")
    print(f"🎯 T=100 平均路径: {cost_T100 / n_tests:.4f} (考量最终寻优极限)")

# ==========================================
# 巅峰对决正式开始
# ==========================================
evaluate_model(net_orig, "原始版 DeepACO (12层GCN)")
evaluate_model(net_gat, "改进版 DeepACO (4层GATv2)")

正在加载 100 个 TSP20 测试实例...
-> 正在加载【原始模型 (12层GCN)】...
-> 正在加载【改进模型 (4层GATv2)】...

🚀 考试开始: 原始版 DeepACO (12层GCN)
   进度: 20 / 100
   进度: 40 / 100
   进度: 60 / 100
   进度: 80 / 100
   进度: 100 / 100

✅ 【原始版 DeepACO (12层GCN)】测试报告单:
⏱️ 总耗时: 289.26 秒
🎯 T=1   平均路径: 3.9257 (考量先验知识质量)
🎯 T=10  平均路径: 3.8273 (考量早期收敛速度)
🎯 T=100 平均路径: 3.8152 (考量最终寻优极限)

🚀 考试开始: 改进版 DeepACO (4层GATv2)
   进度: 20 / 100
   进度: 40 / 100
   进度: 60 / 100
   进度: 80 / 100
   进度: 100 / 100

✅ 【改进版 DeepACO (4层GATv2)】测试报告单:
⏱️ 总耗时: 272.83 秒
🎯 T=1   平均路径: 4.0065 (考量先验知识质量)
🎯 T=10  平均路径: 3.8508 (考量早期收敛速度)
🎯 T=100 平均路径: 3.8185 (考量最终寻优极限)
